## Проверка модели

In [1]:
import torch
from ml.models import model_detector, model_detector_code

model_detector.to('cuda')
model_detector.eval()

x1, x2 = model_detector(torch.randn(1, 1, 640, 640).to('cuda'))
print(x1.shape)
print(x2.keys())


CRITICAL:root:.env.prod


torch.Size([1, 84, 8400])
dict_keys(['boxes', 'scores', 'feats'])


## Гиперпарамтеры

In [2]:
from ml.config import WORKDIR
from ml.models import model_detector
from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
from datetime import datetime

YoloLoss = model_detector.loss # CIoU loss для bboxes; BCELoss - для классов; objBCE - лосс "наличия объекта в bbox"
opt = AdamW(model_detector.parameters(), lr=0.01, weight_decay=5e-4)
lr_sched = MultiStepLR(opt, milestones=[10, 35, 50], gamma=0.1)

epochs = 1
batch_size_train = 64
batch_size_val = 16
batch_size_test = 32 # + drop_last
dataload_workers = 8

threshold_loss = 0.03
early_stopping = 12
models_dir = WORKDIR / 'ml' / 'model_weights' / 'detector' / f'{datetime.now().strftime("%Y%m%d-%H%M%S")}'
models_dir.mkdir(parents=True)


#### Функции для графиков по лоссам и метрикам

In [3]:
from ml.utils import plot_curves


def plot_validation_metrics(history_arg, save_path=None):
    curves = [
        {
            "label": "Val Loss",
            "values": history_arg["val_loss"],
            "color": "red"
        },
        {
            "label": "mAP@0.5",
            "values": history_arg["map50"],
            "color": "blue"
        },
        {
            "label": "mAP@0.5:0.95",
            "values": history_arg["map5095"],
            "color": "green"
        }
    ]

    plot_curves(
        curves=curves,
        title="Validation Loss & Metrics",
        ylabel="Loss / mAP",
        save_path=save_path
    )

def plot_training_dynamics(history_arg, save_path=None):
    curves = [
        {
            "label": "Train Loss",
            "values": history_arg["train_loss"],
            "color": "orange"
        },
        {
            "label": "Val Loss",
            "values": history_arg["val_loss"],
            "color": "red"
        },
        {
            "label": "Learning Rate",
            "values": history_arg["lr"],
            "color": "purple"
        }
    ]

    plot_curves(
        curves=curves,
        title="Training Dynamics",
        ylabel="Loss / LR",
        save_path=save_path
    )


#### Датасет, Даталоадеры | Данные ещё не распределены

In [4]:
from ml.dataclass_detector import OCRDetectorDataset
from torch.utils.data import DataLoader
from ml.config import WORKDIR


train_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'train', 'train')
val_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'val', 'val')
test_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'test', 'val')

train_loader = DataLoader(
    dataset=train_dset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)
val_loader = DataLoader(
    dataset=val_dset,
    batch_size=batch_size_val,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    dataset=val_dset,
    batch_size=batch_size_test,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)

WARNING  | D23-01-2026 T23:42:42 | ml\dataclass_detector.py: def create_data(): line - 163 В наборе данных 960 размеченных изображений
WARNING  | D23-01-2026 T23:42:42 | ml\dataclass_detector.py: def create_data(): line - 163 В наборе данных 240 размеченных изображений
WARNING  | D23-01-2026 T23:42:42 | ml\dataclass_detector.py: def create_data(): line - 163 В наборе данных 326 размеченных изображений


## Обучение & Валидация

In [5]:
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.metrics import ap_per_class, box_iou
from ml.config import env
from tqdm import tqdm
from ml.models import model_detector

model_detector.to(env.device)


print(f'\033[34mОбучение началось\033[0m | Эпохи: \033[33m{epochs}\033[0m')


train_loss, val_loss, lr_list, metrics, min_loss = [], [], [], [], 1.0
plateau_loss_epochs = 0
iouv = torch.linspace(0.5, 0.95, 10).to(env.device)

for epoch in range(1, epochs + 1):

    model_detector.train()

    last_losses_train = []
    train_loop = tqdm(train_loader, leave=False, desc=f'Training \033[34m#{epoch}\033[0m')
    for img, targets in train_loop:
        "Предсказание модели на батч"
        img = img.to(env.device, non_blocking=True)
        targets = targets.to(env.device)

        total_loss_tsr, losses_tsr = YoloLoss(img, targets)

        opt.zero_grad()
        total_loss_tsr.backward()
        opt.step()

        "Считаем лосс"
        total_loss = total_loss_tsr.item()
        losses = [(loss_param, item.item()) for loss_param, item in losses_tsr.items()]
        losses.append(total_loss)
        last_losses_train = losses

        train_loss.append(losses)

    print(f"\033[32mTRAINING\033[0m | Epoch {epoch}, train_loss={last_losses_train[-1]:.4f}, total_loss={last_losses_train:.4f}, {losses[0][0]}_loss={last_losses_train[0][1]:.4f}, {losses[1][0]}_loss={last_losses_train[1][1]:.4f}, {last_losses_train[2][0]}_loss={last_losses_train[2][1]:.4f}")
    train_loss.append(last_losses_train)


    model_detector.eval()
    with torch.no_grad():
        last_losses_val = []
        stats = []

        val_loop = tqdm(val_loader, leave=False, desc=f'\033[Validation \033[36m#{epoch}\033[0m')
        for img, targets in val_loader:

            img = img.to(env.device, non_blocking=True)
            targets = [
                {'boxes': t['boxes'].to(env.device, non_blocking=True),
                'labels': t['labels'].to(env.device, non_blocking=True)}
                for t in targets
            ]

            preds = model_detector(img)
            total_loss_tsr, losses_tsr = YoloLoss(preds, targets)

            total_loss = total_loss_tsr.item()
            losses = [(loss_param, item.item()) for loss_param, item in losses_tsr.items()]
            losses.append(total_loss)
            last_losses_val = losses

            preds_nms = non_max_suppression(
                preds,
                conf_thres=0.25,
                iou_thres=0.45,
                max_det=300
            )

            # Метрики
            for i, pred in enumerate(preds_nms):
                gt = targets[i]  # [num_gt, 5] -> xyxy + cls

                nl = gt.shape[0]
                npr = pred.shape[0]

                correct = torch.zeros(npr, len(iouv), dtype=torch.bool, device=env.device)

                if npr == 0:
                    if nl:
                        stats.append((
                            correct,
                            torch.zeros(0, device=env.device),
                            torch.zeros(0, device=env.device),
                            gt[:, 4]
                        ))
                    continue

                if nl:
                    iou = box_iou(pred[:, :4], gt[:, :4])
                    for j in range(len(iouv)):
                        correct[:, j] = (iou >= iouv[j]).any(1)

                stats.append((
                    correct,
                    pred[:, 4],   # confidence
                    pred[:, 5],   # pred class
                    gt[:, 4]      # gt class
                ))

    lr_sched.step()
    lr = lr_sched.get_last_lr()
    lr_list.append(lr)

    "Подсчёт метрик и лоссов за эпоху"
    total_losses = [all_loss[-1] for all_loss in val_loss]
    val_loss_mean = sum(total_losses) / len(total_losses)

    if len(stats):
        stats = [torch.cat(x, 0).cpu().numpy() for x in zip(*stats)]

        if stats[0].any():
            p, r, ap, f1, ap_class = ap_per_class(*stats)
            map50 = ap[:, 0].mean()
            map5095 = ap.mean()
        else:
            map50, map5095 = 0.0, 0.0

    else:
        map50, map5095 = 0.0, 0.0

    print(f"\033[34mVALIDATION\033[0m | loss={val_loss_mean:.4f} | mAP@0.5={map50:.4f} | mAP@0.5:0.95={map5095:.4f}")

    history = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "map50": map50,
        "map5095": map5095,
        "lr": lr_list
    }

    if min_loss - min_loss * threshold_loss > last_losses_val[-1]:
        min_loss = last_losses_val[-1]

        checkpoint = {
            'model_detector': model_detector_code,
            'state_model': model_detector.state_dict(),
            'state_opt': opt.state_dict(),
            'state_lr_scheduler': lr_sched.state_dict(),
            'history': history,
            'save_epoch': epoch
        }
        torch.save(checkpoint, models_dir.joinpath(f'model_detector{epoch}.pth'))
        plateau_loss_epochs = 0
        print(f'На эпохе - \033[35m{epoch}\033[0m модель сохранена | Ранняя остановка обучения изменена', end='\n\n')

    "Early Stopping"
    if plateau_loss_epochs >= early_stopping:
        print(f'\033[31m{'!!!' * 10} Принудительная остановка обучения, нет прогресса {'!!!' * 10}\033[0m')
        raise Exception("Early Stopping")

    plateau_loss_epochs += 1

plot_training_dynamics(history, models_dir / 'training.png')
plot_validation_metrics(history, models_dir / 'validation.png')

print(f'{'>>>' * 10} Обучение завершено {'<<<' * 10}')

Обучение началось | Эпохи: 1


KeyError: 'boxes'